# Inventory Stockout Prevention Agent
**Generative AI Project | Yiyun Wu, Wanqian Xiong, Angela Kim**

A LangGraph-based procurement agent that autonomously detects stockout risks and generates specific reorder recommendations.

---

## 0. Setup & Dependencies

In [1]:
# Install dependencies (run once)
!pip install langgraph langchain-anthropic anthropic -q

In [2]:
import os
import sqlite3
import json
import math
import time
import logging
from datetime import datetime, timedelta
from typing import TypedDict, Annotated, Optional, Literal
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeoutError

import anthropic
from anthropic import Anthropic
from langgraph.graph import StateGraph, END
from google.colab import userdata

# ── API Key (Colab Secrets) ───────────────────────────────────────────────────
# Add your key in Colab: click the 🔑 icon → New secret → Name: ANTHROPIC_API_KEY
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY") #replace with your api key name
client = Anthropic()

# ── Logging / Telemetry ───────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger("inventory_agent")

print("✅ Setup complete")

/usr/local/lib/python3.12/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


✅ Setup complete


---
## 1. Synthetic SQLite Database
500 SKUs across 5 categories. 5 are seeded below the reorder threshold to guarantee stockout signals.

In [3]:
import random
random.seed(42)

DB_PATH = "inventory.db"

def build_database():
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()

    # ── Schema ────────────────────────────────────────────────────────────────
    c.executescript("""
    DROP TABLE IF EXISTS inventory;
    DROP TABLE IF EXISTS sales_history;
    DROP TABLE IF EXISTS suppliers;

    CREATE TABLE inventory (
        sku         TEXT PRIMARY KEY NOT NULL,
        name        TEXT NOT NULL,
        category    TEXT NOT NULL CHECK(category IN ('electronics','home_goods','apparel','sports','beauty')),
        current_stock INTEGER NOT NULL,
        supplier_id TEXT NOT NULL
    );

    CREATE TABLE sales_history (
        sku         TEXT NOT NULL,
        sale_date   TEXT NOT NULL,
        units_sold  INTEGER NOT NULL,
        PRIMARY KEY (sku, sale_date)
    );

    CREATE TABLE suppliers (
        supplier_id   TEXT PRIMARY KEY NOT NULL,
        name          TEXT NOT NULL,
        lead_time_days INTEGER NOT NULL,
        min_order_qty  INTEGER NOT NULL,
        cost_per_unit  REAL NOT NULL
    );
    """)

    # ── Suppliers (6 suppliers) ───────────────────────────────────────────────
    suppliers = [
        ("SUP-01", "ElectroParts Co",    10, 100, 18.50),
        ("SUP-02", "HomeBase Supply",     7, 50,   8.20),
        ("SUP-03", "FashionFlow Inc",    14, 200,  12.75),
        ("SUP-04", "SportLine Direct",    5, 100,  22.00),
        ("SUP-05", "BeautyWorks Ltd",     8, 150,   6.50),
        ("SUP-12", "TechParts Global",   10, 100,  18.50),
    ]
    c.executemany("INSERT INTO suppliers VALUES (?,?,?,?,?)", suppliers)

    # ── Seeded critical SKUs (below reorder threshold) ────────────────────────
    critical = [
        ("SKU-4412", "Wireless Earbuds",       "electronics", 340,  "SUP-12"),
        ("SKU-2271", "Foam Yoga Mat",           "sports",      180,  "SUP-04"),
        ("SKU-0091", "Retinol Night Cream",     "beauty",      210,  "SUP-05"),
        ("SKU-1133", "Stainless Water Bottle",  "home_goods",  95,   "SUP-02"),
        ("SKU-3302", "Compression Leggings",    "apparel",     320,  "SUP-03"),
    ]
    c.executemany("INSERT INTO inventory VALUES (?,?,?,?,?)", critical)

    # ── Remaining 495 healthy SKUs ────────────────────────────────────────────
    categories = ["electronics","home_goods","apparel","sports","beauty"]
    supplier_map = {
        "electronics": "SUP-01", "home_goods": "SUP-02",
        "apparel": "SUP-03",     "sports": "SUP-04", "beauty": "SUP-05"
    }
    names_pool = {
        "electronics": ["Smart Speaker","USB Hub","LED Monitor","Webcam","Keyboard","Mouse","Charger","Tablet Stand"],
        "home_goods":  ["Coffee Mug","Candle Set","Throw Blanket","Wall Clock","Storage Box","Cutting Board"],
        "apparel":     ["Running Shorts","Denim Jacket","Wool Scarf","Polo Shirt","Sport Socks","Beanie Hat"],
        "sports":      ["Resistance Band","Jump Rope","Kettlebell","Foam Roller","Pull-up Bar","Weight Gloves"],
        "beauty":      ["Face Serum","Lip Balm","Hair Mask","Eye Cream","Sunscreen SPF50","Toner Spray"],
    }
    existing_skus = {r[0] for r in critical}
    i = 0
    while len(existing_skus) < 500:
        sku = f"SKU-{i:04d}"
        if sku not in existing_skus:
            cat = categories[i % 5]
            name = random.choice(names_pool[cat]) + f" #{i}"
            stock = random.randint(500, 2000)  # healthy stock
            c.execute("INSERT INTO inventory VALUES (?,?,?,?,?)",
                      (sku, name, cat, stock, supplier_map[cat]))
            existing_skus.add(sku)
        i += 1

    # ── 90-day sales history ──────────────────────────────────────────────────
    # Critical SKUs: high velocity (triggers stockout detection)
    critical_velocity = {
        "SKU-4412": (42, 6),   # mean=42/day, std=6
        "SKU-2271": (38, 5),
        "SKU-0091": (29, 4),
        "SKU-1133": (22, 3),
        "SKU-3302": (25, 4),
    }
    today = datetime.today().date()
    for sku, (mean, std) in critical_velocity.items():
        for d in range(90):
            sale_date = str(today - timedelta(days=d))
            units = max(1, int(random.gauss(mean, std)))
            c.execute("INSERT OR IGNORE INTO sales_history VALUES (?,?,?)", (sku, sale_date, units))

    # Healthy SKUs: low velocity
    for row in c.execute("SELECT sku FROM inventory").fetchall():
        sku = row[0]
        if sku not in critical_velocity:
            for d in range(90):
                sale_date = str(today - timedelta(days=d))
                units = max(0, int(random.gauss(5, 2)))
                c.execute("INSERT OR IGNORE INTO sales_history VALUES (?,?,?)", (sku, sale_date, units))

    conn.commit()
    conn.close()
    print(f"✅ Database built: 500 SKUs, 90-day history → {DB_PATH}")

build_database()

✅ Database built: 500 SKUs, 90-day history → inventory.db


---
## 2. Tool Implementations

In [4]:
import random as _random

TOOL_TIMEOUT = 10  # seconds

# ── Service Level Tiers (addresses professor feedback #2) ────────────────────
# Dynamic safety stock coefficient by category and margin tier
SERVICE_LEVEL_TIERS = {
    "electronics": {"coefficient": 2.05, "pct": "98%", "reason": "high-margin, high-velocity"},
    "beauty":      {"coefficient": 2.05, "pct": "98%", "reason": "high-margin, perishable"},
    "sports":      {"coefficient": 1.65, "pct": "95%", "reason": "standard"},
    "home_goods":  {"coefficient": 1.65, "pct": "95%", "reason": "standard"},
    "apparel":     {"coefficient": 1.28, "pct": "90%", "reason": "seasonal, high-turnover clearance"},
}
DEFAULT_SERVICE_LEVEL = {"coefficient": 1.65, "pct": "95%", "reason": "default"}

def get_service_level(category: str) -> dict:
    return SERVICE_LEVEL_TIERS.get(category, DEFAULT_SERVICE_LEVEL)


# ── Retry wrapper with exponential backoff (addresses professor feedback #1) ──
def _retry_with_backoff(fn, *args, max_retries=3, base_delay=1.0, timeout=TOOL_TIMEOUT):
    """
    Executes fn(*args) with:
    - Hard timeout per attempt (database/network hang protection)
    - Exponential backoff retry for transient failures (rate limits, connectivity)
    - Distinguishes retryable errors (ConnectionError, TimeoutError, RateLimitError)
      from permanent errors (SKU not found) — permanent errors are not retried.
    """
    from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeout
    import anthropic

    last_error = None
    for attempt in range(max_retries):
        try:
            with ThreadPoolExecutor(max_workers=1) as ex:
                future = ex.submit(fn, *args)
                return future.result(timeout=timeout)
        except FuturesTimeout:
            last_error = f"Tool timed out after {timeout}s (attempt {attempt+1}/{max_retries})"
            logger.warning(last_error)
        except anthropic.RateLimitError as e:
            last_error = f"API rate limit (attempt {attempt+1}/{max_retries}): {e}"
            logger.warning(last_error)
        except anthropic.APIStatusError as e:
            if e.status_code in (500, 502, 503, 529):  # LLM downtime / overload
                last_error = f"API downtime {e.status_code} (attempt {attempt+1}/{max_retries})"
                logger.warning(last_error)
            else:
                return {"error": f"Non-retryable API error {e.status_code}: {e.message}"}
        except sqlite3.OperationalError as e:
            last_error = f"DB connectivity error (attempt {attempt+1}/{max_retries}): {e}"
            logger.warning(last_error)
        except Exception as e:
            # Non-retryable (e.g. SKU not found logic — handled inside the fn)
            return {"error": str(e)}

        if attempt < max_retries - 1:
            delay = base_delay * (2 ** attempt) + _random.uniform(0, 0.5)
            logger.info(f"Retrying in {delay:.1f}s...")
            time.sleep(delay)

    return {"error": f"Failed after {max_retries} attempts. Last error: {last_error}"}


def _run_with_timeout(fn, *args, timeout=TOOL_TIMEOUT):
    """Legacy wrapper — routes through retry logic."""
    return _retry_with_backoff(fn, *args, timeout=timeout)


# ── Data validation helper (addresses professor feedback #3) ─────────────────
def _validate_stock_data(sku: str, current_stock: int, units_sold_7d: int,
                          sales_history: list) -> dict:
    """
    Detects data anomalies before they corrupt downstream calculations.
    Returns dict with 'valid': bool and 'warnings': list of issues found.
    """
    warnings = []

    # Check 1: Negative or zero inventory (data pipeline corruption)
    if current_stock < 0:
        return {"valid": False, "warnings": [f"INVALID: negative stock ({current_stock}) — data corruption suspected"]}
    if current_stock == 0 and units_sold_7d == 0:
        warnings.append("WARNING: Zero stock and zero sales — possible inactive SKU or data feed failure")

    # Check 2: Stale data — no sales recorded in last 14 days
    if units_sold_7d == 0 and current_stock > 0:
        warnings.append("WARNING: Zero sales in 7 days despite positive stock — possible data pipeline gap")

    # Check 3: Demand spike anomaly — flag if recent 7d avg > 3σ above 30d mean
    if len(sales_history) >= 14:
        recent = sales_history[:7]
        baseline = sales_history[7:]
        import statistics as _stats
        if len(baseline) > 1:
            baseline_mean = _stats.mean(baseline)
            baseline_std = _stats.stdev(baseline)
            recent_mean = sum(recent) / len(recent)
            if baseline_std > 0 and (recent_mean - baseline_mean) > 3 * baseline_std:
                warnings.append(
                    f"ANOMALY: Demand spike detected — recent avg {recent_mean:.1f}/day "
                    f"vs baseline {baseline_mean:.1f}±{baseline_std:.1f} (>{3}σ). "
                    "Forecast may be inflated. Verify before ordering."
                )
            # Check 4: Sudden zero — possible stockout masking
            if recent_mean == 0 and baseline_mean > 5:
                warnings.append(
                    f"ANOMALY: Sales dropped to zero (baseline was {baseline_mean:.1f}/day) "
                    "— possible unreported stockout or data gap"
                )

    return {"valid": True, "warnings": warnings}


# ── Tool 1: get_stock_snapshot ────────────────────────────────────────────────
def get_stock_snapshot(sku: str, date: str) -> dict:
    def _query():
        conn = sqlite3.connect(DB_PATH, timeout=5)
        c = conn.cursor()
        row = c.execute(
            "SELECT i.current_stock, i.category, i.supplier_id, i.name "
            "FROM inventory i WHERE i.sku = ?", (sku,)
        ).fetchone()
        if row is None:
            conn.close()
            return {"error": f"SKU '{sku}' not found in inventory."}
        current_stock, category, supplier_id, name = row

        cutoff = str(datetime.strptime(date, "%Y-%m-%d").date() - timedelta(days=7))
        sold = c.execute(
            "SELECT COALESCE(SUM(units_sold),0) FROM sales_history "
            "WHERE sku = ? AND sale_date > ?", (sku, cutoff)
        ).fetchone()[0]

        # Fetch 30-day history for anomaly detection
        history_rows = c.execute(
            "SELECT units_sold FROM sales_history WHERE sku = ? "
            "ORDER BY sale_date DESC LIMIT 30", (sku,)
        ).fetchall()
        conn.close()

        history = [r[0] for r in history_rows]
        validation = _validate_stock_data(sku, current_stock, sold, history)

        return {
            "sku": sku, "sku_name": name, "current_stock": current_stock,
            "units_sold_7d": sold, "category": category, "supplier_id": supplier_id,
            "data_valid": validation["valid"],
            "data_warnings": validation["warnings"]
        }
    return _retry_with_backoff(_query)


# ── Tool 2: calculate_demand_forecast ────────────────────────────────────────
def calculate_demand_forecast(sku: str, horizon: int) -> dict:
    def _query():
        conn = sqlite3.connect(DB_PATH, timeout=5)
        c = conn.cursor()
        rows = c.execute(
            "SELECT units_sold FROM sales_history WHERE sku = ? "
            "ORDER BY sale_date DESC LIMIT 30", (sku,)
        ).fetchall()
        conn.close()
        if not rows:
            return {"error": f"No sales history for '{sku}'."}
        sales = [r[0] for r in rows]
        n = len(sales)
        avg = sum(sales) / n
        first_half = sum(sales[n//2:]) / (n - n//2)
        second_half = sum(sales[:n//2]) / (n//2)
        trend = round((second_half - first_half) / max(first_half, 1), 3)
        seasonality = 1.08 if sku.startswith("SKU-44") or sku.startswith("SKU-00") else 1.00
        std = math.sqrt(sum((s - avg)**2 for s in sales) / n)
        return {
            "sku": sku, "horizon_days": horizon,
            "avg_daily_demand": round(avg, 2),
            "trend_coeff": trend,
            "seasonality_adj": seasonality,
            "demand_std": round(std, 2)
        }
    return _retry_with_backoff(_query)


# ── Tool 3: get_lead_time ─────────────────────────────────────────────────────
def get_lead_time(supplier_id: str) -> dict:
    def _query():
        conn = sqlite3.connect(DB_PATH, timeout=5)
        row = conn.execute(
            "SELECT lead_time_days, min_order_qty, cost_per_unit, name "
            "FROM suppliers WHERE supplier_id = ?", (supplier_id,)
        ).fetchone()
        conn.close()
        if row is None:
            return {"error": f"Supplier '{supplier_id}' not found."}
        return {
            "supplier_id": supplier_id, "supplier_name": row[3],
            "lead_time_days": row[0], "min_order_qty": row[1], "cost_per_unit": row[2]
        }
    return _retry_with_backoff(_query)


# ── Tool 4: calculate_reorder_quantity (now dynamic service level) ────────────
def calculate_reorder_quantity(sku: str, avg_daily_demand: float, lead_time_days: int,
                                demand_std: float, min_order_qty: int,
                                cost_per_unit: float, category: str = "standard") -> dict:
    """
    EOQ-based reorder quantity with DYNAMIC service level by category/margin tier.
    - Electronics/Beauty: 98% service level (z=2.05) — high margin
    - Sports/Home Goods:  95% service level (z=1.65) — standard
    - Apparel:            90% service level (z=1.28) — seasonal clearance
    """
    tier = get_service_level(category)
    z = tier["coefficient"]
    safety_stock = math.ceil(z * demand_std * math.sqrt(lead_time_days))
    raw_qty = math.ceil(avg_daily_demand * lead_time_days) + safety_stock
    reorder_qty = math.ceil(raw_qty / min_order_qty) * min_order_qty
    estimated_cost = round(reorder_qty * cost_per_unit, 2)
    return {
        "sku": sku,
        "service_level": tier["pct"],
        "service_level_reason": tier["reason"],
        "z_coefficient": z,
        "safety_stock": safety_stock,
        "raw_reorder_qty": raw_qty,
        "reorder_qty": reorder_qty,
        "estimated_cost": estimated_cost,
        "formula": (
            f"safety_stock = {z} × {demand_std} × √{lead_time_days} = {safety_stock} | "
            f"reorder = ({avg_daily_demand:.1f} × {lead_time_days}) + {safety_stock} = {raw_qty} → {reorder_qty}"
        )
    }


# ── Tool 5: get_all_at_risk_skus ──────────────────────────────────────────────
def get_all_at_risk_skus(top_n: int = 10) -> dict:
    def _query():
        conn = sqlite3.connect(DB_PATH, timeout=5)
        c = conn.cursor()
        cutoff = str(datetime.today().date() - timedelta(days=7))
        rows = c.execute("""
            SELECT i.sku, i.name, i.category, i.current_stock, i.supplier_id,
                   COALESCE(SUM(sh.units_sold), 0) AS sold_7d,
                   s.lead_time_days
            FROM inventory i
            LEFT JOIN sales_history sh ON i.sku = sh.sku AND sh.sale_date > ?
            JOIN suppliers s ON i.supplier_id = s.supplier_id
            GROUP BY i.sku
        """, (cutoff,)).fetchall()
        conn.close()
        ranked = []
        for sku, name, cat, stock, sup, sold_7d, lead in rows:
            avg_daily = sold_7d / 7 if sold_7d > 0 else 0.1
            days_remaining = round(stock / avg_daily, 1)
            shortfall = lead - days_remaining
            if shortfall > 0:
                ranked.append({
                    "sku": sku, "name": name, "category": cat,
                    "days_remaining": days_remaining, "lead_time_days": lead,
                    "shortfall_days": round(shortfall, 1), "supplier_id": sup
                })
        ranked.sort(key=lambda x: x["shortfall_days"], reverse=True)
        return {"total_at_risk": len(ranked), "at_risk": ranked[:top_n]}
    return _retry_with_backoff(_query)


TOOLS = {
    "get_stock_snapshot": get_stock_snapshot,
    "calculate_demand_forecast": calculate_demand_forecast,
    "get_lead_time": get_lead_time,
    "calculate_reorder_quantity": calculate_reorder_quantity,
    "get_all_at_risk_skus": get_all_at_risk_skus,
}

print("Tools defined:", list(TOOLS.keys()))
print("Service level tiers:", {k: v["pct"] for k, v in SERVICE_LEVEL_TIERS.items()})


Tools defined: ['get_stock_snapshot', 'calculate_demand_forecast', 'get_lead_time', 'calculate_reorder_quantity', 'get_all_at_risk_skus']
Service level tiers: {'electronics': '98%', 'beauty': '98%', 'sports': '95%', 'home_goods': '95%', 'apparel': '90%'}


---
## 3. LangGraph State & System Prompts

In [5]:
# ── Agent State ───────────────────────────────────────────────────────────────
class AgentState(TypedDict):
    user_query: str
    parsed_intent: Optional[dict]
    messages: list
    tool_outputs: list
    stock_snapshot: Optional[dict]
    demand_forecast: Optional[dict]
    lead_time_info: Optional[dict]
    reorder_calc: Optional[dict]
    final_recommendation: Optional[str]
    early_exit: bool
    error: Optional[str]
    iteration_count: int
    trace_log: list
    token_usage: dict
    node_timings: dict          # per-node latency for bottleneck analysis
    run_outcome: Optional[str]  # 'success' | 'early_exit' | 'error'
    total_tokens_used: int      # cumulative token count across all LLM calls
    token_budget_exceeded: bool # hard kill switch: True if budget is hit


INTENT_PARSER_PROMPT = (
    'You are an intent parser for an inventory management system.\n'
    'Extract structured information from the user\'s query. Respond ONLY with valid JSON.\n'
    '\n'
    'Output schema:\n'
    '{\n'
    '  "mode": "single_sku" | "scan_all" | "comparison",\n'
    '  "sku": "SKU-XXXX" or null,\n'
    '  "horizon_days": integer (default 7),\n'
    '  "date": "YYYY-MM-DD" (default today),\n'
    '  "category_filter": null | "electronics" | "home_goods" | "apparel" | "sports" | "beauty"\n'
    '}\n'
    '\n'
    'Rules:\n'
    '- If the query mentions a specific SKU code, mode = "single_sku".\n'
    '- If the query is broad ("what do we need to reorder", "anything running low"), mode = "scan_all".\n'
    '- If the query asks to compare two categories, mode = "comparison".\n'
    '- "this week" → horizon_days = 7. "this month" → 30.\n'
    '- Always output today\'s date unless a specific date is mentioned.\n'
)

WORKER_SYSTEM_PROMPT = (
    'You are an expert procurement AI for a mid-size retailer managing 500 SKUs.\n'
    'Your job is to detect stockout risks and generate specific, evidence-based reorder recommendations.\n'
    '\n'
    'You have access to these tools:\n'
    '- get_stock_snapshot(sku, date): Current stock and 7-day velocity\n'
    '- calculate_demand_forecast(sku, horizon): Demand forecast with trend + seasonality\n'
    '- get_lead_time(supplier_id): Supplier lead time and ordering constraints\n'
    '- calculate_reorder_quantity(sku, avg_daily_demand, lead_time_days, demand_std, min_order_qty, cost_per_unit): EOQ calc\n'
    '- get_all_at_risk_skus(top_n): Scans all SKUs, returns most at-risk ranked by shortfall\n'
    '\n'
    '<reasoning_protocol>\n'
    'For EACH step, explicitly state:\n'
    'THOUGHT: What I need to find out and why.\n'
    'ACTION: Which tool to call with which parameters.\n'
    'OBSERVATION: What the tool returned.\n'
    'CALCULATION: Any math I perform (show your work).\n'
    '\n'
    'After all steps, produce a FINAL RECOMMENDATION with:\n'
    '- Specific SKU, quantity, supplier, and deadline\n'
    '- Estimated cost\n'
    '- Urgency tier (CRITICAL / HIGH / MEDIUM)\n'
    '- Evidence: exact numbers from tool outputs\n'
    '</reasoning_protocol>\n'
    '\n'
    '<guardrails>\n'
    '- NEVER cite inventory numbers not returned by a tool.\n'
    '- If a SKU is not found, respond: "[SKU] not found — no recommendation generated."\n'
    '- If stock is healthy (days_remaining >= lead_time_days), state "No action required" and stop.\n'
    '</guardrails>\n'
)

# Judge prompt: 4 standard dims + 3 domain-specific KPIs (7 dims, max score = 21)
JUDGE_SYSTEM_PROMPT = (
    'You are an expert evaluator for an inventory optimization AI agent.\n'
    'Score the agent\'s response on SEVEN dimensions (0-3 each, max total = 21).\n'
    '\n'
    'STANDARD DIMENSIONS:\n'
    '1. instruction_adherence: Did it answer the specific inventory question asked?\n'
    '   0=ignored, 1=partial, 2=mostly, 3=fully addressed\n'
    '\n'
    '2. reasoning_transparency: Is each conclusion backed by explicit tool call outputs?\n'
    '   0=black box, 1=some numbers, 2=most steps shown, 3=fully transparent ReAct trace\n'
    '\n'
    '3. hallucination_check: Does it cite figures NOT present in tool outputs?\n'
    '   0=clear hallucination, 1=suspicious claim, 2=minor ambiguity, 3=fully grounded\n'
    '\n'
    '4. recommendation_specificity: qty + urgency tier + deadline + cost evidence?\n'
    '   0=vague, 1=qty only, 2=qty+tier, 3=qty+tier+deadline+cost+evidence\n'
    '\n'
    'DOMAIN-SPECIFIC KPIs:\n'
    '5. reorder_qty_accuracy: Is the recommended qty consistent with the EOQ formula output?\n'
    '   Compare agent qty vs calculate_reorder_quantity tool output.\n'
    '   0=no qty stated, 1=>20% off formula, 2=within 20%, 3=within 5% or exact match\n'
    '\n'
    '6. false_alarm_check: If stock was healthy, did agent correctly avoid recommending a reorder?\n'
    '   0=wrongly recommended reorder on healthy stock, 3=correctly said no action OR stock was genuinely at risk\n'
    '\n'
    '7. urgency_tier_correctness: Is CRITICAL/HIGH/MEDIUM consistent with shortfall severity?\n'
    '   CRITICAL=shortfall>5 days, HIGH=2-5 days, MEDIUM=<2 days.\n'
    '   0=wrong tier, 1=off by one tier, 2=close, 3=correct tier (or N/A → score 3)\n'
    '\n'
    'Respond ONLY with valid JSON — no preamble, no markdown:\n'
    '{"instruction_adherence": int, "reasoning_transparency": int, "hallucination_check": int,\n'
    ' "recommendation_specificity": int, "reorder_qty_accuracy": int, "false_alarm_check": int,\n'
    ' "urgency_tier_correctness": int, "total_score": int, "reasoning": "one sentence"}\n'
)

print('✅ State schema and prompts ready (7-dim judge with 3 domain-specific KPIs)')


✅ State schema and prompts ready (7-dim judge with 3 domain-specific KPIs)


---
## 4. LangGraph Nodes

In [6]:
MAX_ITERATIONS = 6
TODAY = str(datetime.today().date())

# ── Token Budget Kill Switch (FinOps guardrail) ───────────────────────────────
# Hard ceiling on cumulative tokens per agent run to prevent runaway costs.
# At Sonnet pricing (~$3/M input, $15/M output), 50K tokens ≈ $0.15 max per run.
TOKEN_BUDGET = 50_000  # cumulative input + output tokens across all LLM calls

def _check_token_budget(state: AgentState) -> bool:
    """Return True if cumulative token usage has exceeded TOKEN_BUDGET."""
    return state.get("total_tokens_used", 0) >= TOKEN_BUDGET

def _log_trace(state, node, detail):
    entry = {"ts": datetime.now().isoformat(), "node": node, "detail": detail}
    logger.info(f"[{node}] {detail}")
    return state["trace_log"] + [entry]

def _add_tokens(state, role, input_tokens, output_tokens):
    usage = dict(state["token_usage"])
    usage[role] = usage.get(role, {"input": 0, "output": 0})
    usage[role]["input"] += input_tokens
    usage[role]["output"] += output_tokens
    return usage

def _add_tokens_and_check_budget(state, role, input_tokens, output_tokens):
    """Update token usage AND check cumulative budget. Returns (new_usage, new_total, budget_hit)."""
    usage = _add_tokens(state, role, input_tokens, output_tokens)
    new_total = state.get("total_tokens_used", 0) + input_tokens + output_tokens
    budget_hit = new_total >= TOKEN_BUDGET
    if budget_hit:
        logger.warning(f"TOKEN BUDGET EXCEEDED: {new_total} tokens used (limit={TOKEN_BUDGET})")
    return usage, new_total, budget_hit

def _record_timing(state, node, elapsed):
    timings = dict(state.get("node_timings", {}))
    timings[node] = round(elapsed, 3)
    return timings

def _format_tool_outputs(tool_outputs):
    """Format tool outputs as a readable block for LLM prompts."""
    lines = []
    for tool_name, result in tool_outputs:
        lines.append(f"TOOL: {tool_name}")
        if isinstance(result, dict):
            for k, v in result.items():
                lines.append(f"  {k}: {v}")
        else:
            lines.append(f"  result: {result}")
        lines.append("")
    return "\n".join(lines)


# ── Node 1: Intent Parser ─────────────────────────────────────────────────────
INTENT_PARSER_PROMPT = (
    'You are an intent parser for an inventory management system.\n'
    'Extract structured information from the user\'s query. Respond ONLY with valid JSON, no markdown fences.\n'
    '\n'
    'Output schema:\n'
    '{\n'
    '  "mode": "single_sku" | "scan_all" | "comparison",\n'
    '  "sku": "SKU-XXXX" or null,\n'
    '  "skus": ["SKU-XXXX", "SKU-YYYY"] or null,\n'
    '  "horizon_days": integer (default 7),\n'
    '  "date": "YYYY-MM-DD" (default today),\n'
    '  "category_filter": null | "electronics" | "home_goods" | "apparel" | "sports" | "beauty"\n'
    '}\n'
    '\n'
    'Rules:\n'
    '- If the query mentions a specific single SKU code, mode = "single_sku" and populate "sku".\n'
    '- If the query asks to compare two specific SKUs, mode = "comparison" and populate "skus" list.\n'
    '- If the query is broad ("what do we need to reorder", "anything running low"), mode = "scan_all".\n'
    '- "this week" -> horizon_days = 7. "this month" -> 30.\n'
    '- Always output today\'s date unless a specific date is mentioned.\n'
    '- Respond ONLY with the JSON object. No markdown, no explanation.\n'
)

def intent_parser_node(state: AgentState) -> AgentState:
    t0 = time.time()
    trace = _log_trace(state, "IntentParser", "Parsing: " + state["user_query"])
    try:
        resp = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=300,
            system=INTENT_PARSER_PROMPT,
            messages=[{"role": "user", "content": state["user_query"]}]
        )
        raw = resp.content[0].text.strip()
        # Strip markdown code fences if present
        if raw.startswith("```"):
            raw = raw.split("```")[1]
            if raw.startswith("json"):
                raw = raw[4:]
        raw = raw.strip()
        intent = json.loads(raw)
        if "date" not in intent or not intent["date"]:
            intent["date"] = TODAY
        token_usage, new_total, budget_hit = _add_tokens_and_check_budget(
            state, "intent_parser", resp.usage.input_tokens, resp.usage.output_tokens)
        trace = _log_trace({**state, "trace_log": trace}, "IntentParser",
                           f"Parsed intent: {intent} | cumulative_tokens={new_total}")
        return {**state, "parsed_intent": intent, "trace_log": trace,
                "token_usage": token_usage, "total_tokens_used": new_total,
                "token_budget_exceeded": budget_hit, "error": None,
                "node_timings": _record_timing(state, "IntentParser", time.time() - t0)}
    except Exception as e:
        fallback = {"mode": "scan_all", "sku": None, "skus": None, "horizon_days": 7,
                    "date": TODAY, "category_filter": None}
        trace = _log_trace({**state, "trace_log": trace}, "IntentParser",
                           f"Parse failed ({e}), falling back to scan_all")
        return {**state, "parsed_intent": fallback, "trace_log": trace, "error": str(e),
                "node_timings": _record_timing(state, "IntentParser", time.time() - t0)}


# ── Node 2: Stock Snapshot SQL ────────────────────────────────────────────────
def stock_level_node(state: AgentState) -> AgentState:
    t0 = time.time()
    intent = state["parsed_intent"]
    trace = _log_trace(state, "StockLevel", "mode=" + intent["mode"])

    if intent["mode"] == "scan_all":
        result = get_all_at_risk_skus(top_n=5)
        trace = _log_trace({**state, "trace_log": trace}, "StockLevel",
                           "Scan found " + str(result["total_at_risk"]) + " at-risk SKUs")
        return {**state, "stock_snapshot": result, "trace_log": trace,
                "tool_outputs": state["tool_outputs"] + [("get_all_at_risk_skus", result)],
                "node_timings": _record_timing(state, "StockLevel", time.time() - t0)}

    if intent["mode"] == "comparison":
        skus = intent.get("skus") or []
        results = []
        for sku in skus:
            r = get_stock_snapshot(sku, intent["date"])
            if "error" not in r:
                avg_daily = r["units_sold_7d"] / 7 if r["units_sold_7d"] > 0 else 0.1
                r["avg_daily_approx"] = round(avg_daily, 2)
                r["days_remaining_approx"] = round(r["current_stock"] / avg_daily, 1)
                results.append(r)
        combined = {
            "comparison": results,
            "total_at_risk": len([r for r in results if r.get("days_remaining_approx", 999) < 30])
        }
        trace = _log_trace({**state, "trace_log": trace}, "StockLevel",
                           f"Comparison: {[r['sku'] for r in results]}")
        return {**state, "stock_snapshot": combined, "trace_log": trace,
                "tool_outputs": state["tool_outputs"] + [("get_stock_snapshots_comparison", combined)],
                "node_timings": _record_timing(state, "StockLevel", time.time() - t0)}

    sku = intent.get("sku")
    if not sku:
        return {**state, "error": "No SKU specified and mode is not scan_all.",
                "trace_log": trace,
                "node_timings": _record_timing(state, "StockLevel", time.time() - t0)}

    result = get_stock_snapshot(sku, intent["date"])
    if "error" in result:
        trace = _log_trace({**state, "trace_log": trace}, "StockLevel", f"SKU not found: {sku}")
        return {**state, "stock_snapshot": result, "error": result["error"],
                "trace_log": trace, "early_exit": True,
                "node_timings": _record_timing(state, "StockLevel", time.time() - t0)}

    avg_daily = result["units_sold_7d"] / 7 if result["units_sold_7d"] > 0 else 0.1
    result["avg_daily_approx"] = round(avg_daily, 2)
    result["days_remaining_approx"] = round(result["current_stock"] / avg_daily, 1)
    trace = _log_trace({**state, "trace_log": trace}, "StockLevel",
                       sku + ": stock=" + str(result["current_stock"]) + ", ~" + str(result["days_remaining_approx"]) + " days remaining")
    return {**state, "stock_snapshot": result, "trace_log": trace,
            "tool_outputs": state["tool_outputs"] + [("get_stock_snapshot", result)],
            "node_timings": _record_timing(state, "StockLevel", time.time() - t0)}


# ── Node 2b: Data Validation ─────────────────────────────────────────────────
def data_validation_node(state: AgentState) -> AgentState:
    t0 = time.time()
    snap = state.get("stock_snapshot") or {}

    if "at_risk" in snap:
        if not snap["at_risk"]:
            return {**state, "node_timings": _record_timing(state, "DataValidation", time.time() - t0)}
        warnings = []
    elif "comparison" in snap:
        warnings = []
    else:
        warnings = snap.get("data_warnings", [])
        if not snap.get("data_valid", True):
            msg = "DATA VALIDATION FAILED: " + " | ".join(warnings)
            trace = _log_trace(state, "DataValidation", msg)
            return {**state, "error": msg, "early_exit": True, "trace_log": trace,
                    "node_timings": _record_timing(state, "DataValidation", time.time() - t0)}

    trace = state["trace_log"]
    for w in warnings:
        trace = _log_trace({**state, "trace_log": trace}, "DataValidation", w)

    return {**state, "trace_log": trace,
            "node_timings": _record_timing(state, "DataValidation", time.time() - t0)}


# ── Conditional Edge Router ───────────────────────────────────────────────────
def route_after_stock(state: AgentState) -> Literal["early_exit", "forecasting"]:
    if state.get("early_exit") or state.get("error"):
        return "early_exit"
    snap = state.get("stock_snapshot") or {}
    if "at_risk" in snap:
        return "early_exit" if snap["total_at_risk"] == 0 else "forecasting"
    if "comparison" in snap:
        return "forecasting" if snap["comparison"] else "early_exit"
    days_remaining = snap.get("days_remaining_approx", 0)
    return "early_exit" if days_remaining > 30 else "forecasting"


# ── Node 3: Demand Forecast + Lead Time ──────────────────────────────────────
def forecasting_node(state: AgentState) -> AgentState:
    t0 = time.time()
    trace = _log_trace(state, "Forecasting", "Running demand forecast and lead time lookup")
    snap = state.get("stock_snapshot") or {}
    intent = state["parsed_intent"]

    if "comparison" in snap and snap["comparison"]:
        # Use the most urgent SKU (lowest days_remaining) for detailed forecast
        top = sorted(snap["comparison"], key=lambda x: x.get("days_remaining_approx", 999))[0]
        sku = top["sku"]
        supplier_id = top["supplier_id"]
    elif "at_risk" in snap and snap["at_risk"]:
        top = snap["at_risk"][0]
        sku = top["sku"]
        supplier_id = top["supplier_id"]
    else:
        sku = snap["sku"]
        supplier_id = snap["supplier_id"]

    horizon = intent.get("horizon_days", 7)
    forecast = calculate_demand_forecast(sku, horizon)
    lead = get_lead_time(supplier_id)
    tool_outputs = state["tool_outputs"] + [
        ("calculate_demand_forecast", forecast),
        ("get_lead_time", lead)
    ]
    trace = _log_trace({**state, "trace_log": trace}, "Forecasting",
                       "avg_daily=" + str(forecast.get("avg_daily_demand")) + ", lead=" + str(lead.get("lead_time_days")) + " days")
    return {**state, "demand_forecast": forecast, "lead_time_info": lead,
            "tool_outputs": tool_outputs, "trace_log": trace,
            "node_timings": _record_timing(state, "Forecasting", time.time() - t0)}


# ── Node 4: Reorder Quantity Calculation ─────────────────────────────────────
def reorder_calc_node(state: AgentState) -> AgentState:
    t0 = time.time()
    trace = _log_trace(state, "ReorderCalc", "Computing EOQ reorder quantity")
    forecast = state["demand_forecast"]
    lead = state["lead_time_info"]

    if "error" in forecast or "error" in lead:
        return {**state, "error": "Tool failure in forecasting", "trace_log": trace,
                "node_timings": _record_timing(state, "ReorderCalc", time.time() - t0)}

    sku = forecast["sku"]
    snap = state.get("stock_snapshot") or {}
    if "comparison" in snap and snap["comparison"]:
        top = sorted(snap["comparison"], key=lambda x: x.get("days_remaining_approx", 999))[0]
        category = top.get("category", "standard")
    elif "at_risk" in snap and snap["at_risk"]:
        category = snap["at_risk"][0].get("category", "standard")
    else:
        category = snap.get("category", "standard")

    result = calculate_reorder_quantity(
        sku=sku,
        avg_daily_demand=forecast["avg_daily_demand"],
        lead_time_days=lead["lead_time_days"],
        demand_std=forecast["demand_std"],
        min_order_qty=lead["min_order_qty"],
        cost_per_unit=lead["cost_per_unit"],
        category=category
    )
    tool_outputs = state["tool_outputs"] + [("calculate_reorder_quantity", result)]
    trace = _log_trace({**state, "trace_log": trace}, "ReorderCalc",
                       "Reorder qty=" + str(result["reorder_qty"]) + ", cost=$" + str(result["estimated_cost"]))
    return {**state, "reorder_calc": result, "tool_outputs": tool_outputs, "trace_log": trace,
            "node_timings": _record_timing(state, "ReorderCalc", time.time() - t0)}


# ── Node 5: Synthesizer ───────────────────────────────────────────────────────
def synthesizer_node(state: AgentState) -> AgentState:
    t0 = time.time()
    trace = _log_trace(state, "Synthesizer", "Generating procurement recommendation")

    if state["iteration_count"] >= MAX_ITERATIONS:
        return {**state, "final_recommendation": "MAX_ITERATIONS reached — agent stopped.",
                "trace_log": trace, "run_outcome": "error",
                "node_timings": _record_timing(state, "Synthesizer", time.time() - t0)}

    # ── Token Budget Kill Switch ──────────────────────────────────────────────
    if _check_token_budget(state):
        msg = (f"TOKEN BUDGET EXCEEDED ({state.get('total_tokens_used', 0):,} / {TOKEN_BUDGET:,} tokens). "
               "Agent stopped to prevent runaway cost. Reduce query complexity or raise TOKEN_BUDGET.")
        trace = _log_trace({**state, "trace_log": trace}, "Synthesizer", msg)
        return {**state, "final_recommendation": msg, "trace_log": trace,
                "run_outcome": "error", "token_budget_exceeded": True,
                "node_timings": _record_timing(state, "Synthesizer", time.time() - t0)}

    tool_block = _format_tool_outputs(state["tool_outputs"])

    user_msg = (
        "USER QUERY: " + state["user_query"] + "\n\n"
        "TOOL OUTPUTS (cite ONLY these numbers — do not invent any values):\n"
        + tool_block + "\n"
        "Produce your response in this exact format:\n\n"
        "THOUGHT: What the tool outputs tell you about stock risk.\n"
        "CALCULATION: Show the math using the exact numbers above.\n"
        "FINAL RECOMMENDATION:\n"
        "  SKU: [from tool output]\n"
        "  Order quantity: [from calculate_reorder_quantity.reorder_qty]\n"
        "  Supplier: [from get_lead_time.supplier_name]\n"
        "  Estimated cost: $[from calculate_reorder_quantity.estimated_cost]\n"
        "  Urgency tier: CRITICAL (shortfall>5d) / HIGH (2-5d) / MEDIUM (<2d)\n"
        "  Deadline: Place PO by EOD today\n"
        "  Evidence: [quote specific numbers from the tool outputs above]"
    )

    resp = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1200,
        system=WORKER_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_msg}]
    )
    recommendation = resp.content[0].text.strip()
    token_usage, new_total, budget_hit = _add_tokens_and_check_budget(
        state, "worker", resp.usage.input_tokens, resp.usage.output_tokens)
    trace = _log_trace({**state, "trace_log": trace}, "Synthesizer",
                       f"Recommendation generated ({resp.usage.output_tokens} output tokens) | cumulative_tokens={new_total}")
    return {**state, "final_recommendation": recommendation, "trace_log": trace,
            "token_usage": token_usage, "total_tokens_used": new_total,
            "token_budget_exceeded": budget_hit, "run_outcome": "success",
            "iteration_count": state["iteration_count"] + 1,
            "node_timings": _record_timing(state, "Synthesizer", time.time() - t0)}


# ── Node 6: Early Exit ────────────────────────────────────────────────────────
def early_exit_node(state: AgentState) -> AgentState:
    t0 = time.time()
    if state.get("error") and "not found" in state["error"].lower():
        msg = "WARNING: " + state["error"] + " No recommendation generated."
    else:
        snap = state.get("stock_snapshot") or {}
        days = snap.get("days_remaining_approx", "N/A")
        msg = "Stock healthy (~" + str(days) + " days remaining). No reorder action required."
    trace = _log_trace(state, "EarlyExit", msg)
    return {**state, "final_recommendation": msg, "trace_log": trace,
            "run_outcome": "early_exit",
            "node_timings": _record_timing(state, "EarlyExit", time.time() - t0)}


print("All graph nodes defined")

All graph nodes defined


In [7]:
import inspect
print(inspect.getsource(intent_parser_node))

def intent_parser_node(state: AgentState) -> AgentState:
    t0 = time.time()
    trace = _log_trace(state, "IntentParser", "Parsing: " + state["user_query"])
    try:
        resp = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=300,
            system=INTENT_PARSER_PROMPT,
            messages=[{"role": "user", "content": state["user_query"]}]
        )
        raw = resp.content[0].text.strip()
        # Strip markdown code fences if present
        if raw.startswith("```"):
            raw = raw.split("```")[1]
            if raw.startswith("json"):
                raw = raw[4:]
        raw = raw.strip()
        intent = json.loads(raw)
        if "date" not in intent or not intent["date"]:
            intent["date"] = TODAY
        token_usage, new_total, budget_hit = _add_tokens_and_check_budget(
            state, "intent_parser", resp.usage.input_tokens, resp.usage.output_tokens)
        trace = _log_trace({**state, "trace_log": tr

In [8]:
resp = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=300,
    system=INTENT_PARSER_PROMPT,
    messages=[{"role": "user", "content": "Why is SKU-4412 running low and what should we order?"}]
)
raw = resp.content[0].text.strip()
print(repr(raw))

'```json\n{\n  "mode": "single_sku",\n  "sku": "SKU-4412",\n  "skus": null,\n  "horizon_days": 7,\n  "date": "2025-04-15",\n  "category_filter": null\n}\n```'


---
## 5. Compile the LangGraph StateGraph

In [9]:
builder = StateGraph(AgentState)

# Add nodes
builder.add_node("intent_parser", intent_parser_node)
builder.add_node("stock_level", stock_level_node)
builder.add_node("data_validation", data_validation_node)   # NEW: anomaly detection
builder.add_node("forecasting", forecasting_node)
builder.add_node("reorder_calc", reorder_calc_node)
builder.add_node("synthesizer", synthesizer_node)
builder.add_node("early_exit", early_exit_node)

# Entry point
builder.set_entry_point("intent_parser")

# Linear edges
builder.add_edge("intent_parser", "stock_level")
builder.add_edge("stock_level", "data_validation")          # NEW: validation after stock query
builder.add_edge("forecasting", "reorder_calc")
builder.add_edge("reorder_calc", "synthesizer")
builder.add_edge("synthesizer", END)
builder.add_edge("early_exit", END)

# Conditional edge now fires AFTER data_validation (not after stock_level)
builder.add_conditional_edges(
    "data_validation",
    route_after_stock,
    {"early_exit": "early_exit", "forecasting": "forecasting"}
)

graph = builder.compile()
print("LangGraph compiled — pipeline: intent_parser → stock_level → data_validation → [early_exit | forecasting → reorder_calc → synthesizer]")


LangGraph compiled — pipeline: intent_parser → stock_level → data_validation → [early_exit | forecasting → reorder_calc → synthesizer]


---
## 5b. System Architecture Diagram

The pipeline below reflects the **actual compiled graph** including the `data_validation` node added after professor feedback.


In [10]:
# ── Architecture Diagram (Mermaid) ───────────────────────────────────────────
# Rendered via mermaid.ink — no extra install needed in Colab

MERMAID_DIAGRAM = """
flowchart TD
    A([User Query]) --> B[Intent Parser\nHaiku 4.5\nJSON extraction]
    B --> C[Stock Level SQL\nget_stock_snapshot\nget_all_at_risk_skus]
    C --> D{Data Validation\nAnomaly Detection\nPipeline Check}
    D -->|Invalid data\nor SKU not found| E([Early Exit\nNo hallucination\npossible])
    D -->|Stock healthy\ndays_remaining > 30| E
    D -->|At-risk stock\ndetected| F[Demand Forecast\ncalculate_demand_forecast\nrolling avg + seasonality]
    F --> G[Lead Time Lookup\nget_lead_time\nSQL key-value]
    G --> H[Reorder Qty Calc\nEOQ + dynamic\nservice level z]
    H --> I[Synthesizer\nSonnet 4.6\nReAct trace]
    I --> J([Procurement\nRecommendation])
    J --> K[Judge LLM\nSonnet 4.6\n7-dim rubric]

    style A fill:#4A90D9,color:#fff
    style E fill:#E8504A,color:#fff
    style J fill:#27AE60,color:#fff
    style K fill:#8E44AD,color:#fff
    style B fill:#2C3E50,color:#fff
    style I fill:#2C3E50,color:#fff
    style D fill:#F39C12,color:#fff
"""

import base64, urllib.parse
from IPython.display import Image, display

# Encode and render via mermaid.ink
encoded = base64.urlsafe_b64encode(MERMAID_DIAGRAM.encode()).decode()
url = f"https://mermaid.ink/img/{encoded}"
print("Architecture Diagram:")
print(f"Direct link: {url}")

try:
    display(Image(url=url))
except Exception:
    # Fallback: print the diagram source for manual rendering
    print("\nMermaid source (paste at https://mermaid.live to render):")
    print(MERMAID_DIAGRAM)


Architecture Diagram:
Direct link: https://mermaid.ink/img/CmZsb3djaGFydCBURAogICAgQShbVXNlciBRdWVyeV0pIC0tPiBCW0ludGVudCBQYXJzZXIKSGFpa3UgNC41CkpTT04gZXh0cmFjdGlvbl0KICAgIEIgLS0-IENbU3RvY2sgTGV2ZWwgU1FMCmdldF9zdG9ja19zbmFwc2hvdApnZXRfYWxsX2F0X3Jpc2tfc2t1c10KICAgIEMgLS0-IER7RGF0YSBWYWxpZGF0aW9uCkFub21hbHkgRGV0ZWN0aW9uClBpcGVsaW5lIENoZWNrfQogICAgRCAtLT58SW52YWxpZCBkYXRhCm9yIFNLVSBub3QgZm91bmR8IEUoW0Vhcmx5IEV4aXQKTm8gaGFsbHVjaW5hdGlvbgpwb3NzaWJsZV0pCiAgICBEIC0tPnxTdG9jayBoZWFsdGh5CmRheXNfcmVtYWluaW5nID4gMzB8IEUKICAgIEQgLS0-fEF0LXJpc2sgc3RvY2sKZGV0ZWN0ZWR8IEZbRGVtYW5kIEZvcmVjYXN0CmNhbGN1bGF0ZV9kZW1hbmRfZm9yZWNhc3QKcm9sbGluZyBhdmcgKyBzZWFzb25hbGl0eV0KICAgIEYgLS0-IEdbTGVhZCBUaW1lIExvb2t1cApnZXRfbGVhZF90aW1lClNRTCBrZXktdmFsdWVdCiAgICBHIC0tPiBIW1Jlb3JkZXIgUXR5IENhbGMKRU9RICsgZHluYW1pYwpzZXJ2aWNlIGxldmVsIHpdCiAgICBIIC0tPiBJW1N5bnRoZXNpemVyClNvbm5ldCA0LjYKUmVBY3QgdHJhY2VdCiAgICBJIC0tPiBKKFtQcm9jdXJlbWVudApSZWNvbW1lbmRhdGlvbl0pCiAgICBKIC0tPiBLW0p1ZGdlIExMTQpTb25uZXQgNC42CjctZGltIHJ1YnJpY10KCiAgI

---
## 6. Run Helper + FinOps Reporter

In [11]:
# Pricing (per 1K tokens)
PRICING = {
    'intent_parser':  {'input': 0.001, 'output': 0.005},
    'worker':         {'input': 0.003, 'output': 0.015},
    'judge':          {'input': 0.003, 'output': 0.015},
}

def compute_cost(token_usage: dict) -> dict:
    total = 0.0
    breakdown = {}
    for role, usage in token_usage.items():
        p = PRICING.get(role, {'input': 0.003, 'output': 0.015})
        cost = (usage['input'] / 1000 * p['input']) + (usage['output'] / 1000 * p['output'])
        breakdown[role] = round(cost, 6)
        total += cost
    return {'breakdown': breakdown, 'total': round(total, 6)}


def run_agent(query: str, verbose: bool = True) -> dict:
    """Run the inventory agent on a user query. Returns the final state."""
    initial_state: AgentState = {
        'user_query': query,
        'parsed_intent': None,
        'messages': [],
        'tool_outputs': [],
        'stock_snapshot': None,
        'demand_forecast': None,
        'lead_time_info': None,
        'reorder_calc': None,
        'final_recommendation': None,
        'early_exit': False,
        'error': None,
        'iteration_count': 0,
        'trace_log': [],
        'token_usage': {},
        'node_timings': {},
        'run_outcome': None,
        'total_tokens_used': 0,
        'token_budget_exceeded': False,
    }
    t0 = time.time()
    final_state = graph.invoke(initial_state)
    elapsed = round(time.time() - t0, 2)

    if verbose:
        print('\n' + '═'*70)
        print(f"QUERY: {query}")
        print('═'*70)
        print(final_state['final_recommendation'])
        cost = compute_cost(final_state['token_usage'])
        print('\n' + '─'*70)
        print(f"⏱  Total latency: {elapsed}s | outcome: {final_state.get('run_outcome','?')} | tokens_used: {final_state.get('total_tokens_used',0):,}/{TOKEN_BUDGET:,}")
        print(f"💰 Cost: ${cost['total']:.5f}")
        for role, c in cost['breakdown'].items():
            print(f"   {role}: ${c:.5f}")
        timings = final_state.get('node_timings', {})
        if timings:
            print('⏱  Node latency breakdown:')
            for node, t in timings.items():
                pct = round(t / elapsed * 100) if elapsed > 0 else 0
                bar = '█' * (pct // 5)
                print(f"   {node:<16} {t:.3f}s  {bar} {pct}%")
        print('─'*70)

    final_state['_elapsed'] = elapsed
    return final_state


print('✅ run_agent() ready')


✅ run_agent() ready


---
## 7. Judge LLM (Automated Evaluator)

In [12]:
def judge_response(user_query: str, agent_recommendation: str, tool_outputs: list) -> dict:
    """Evaluate a Worker response with the Judge LLM (7 dimensions, max=21)."""
    # Build a structured tool summary so judge can verify grounding
    tool_lines = []
    for tool_name, result in tool_outputs:
        tool_lines.append(f"TOOL: {tool_name}")
        if isinstance(result, dict):
            for k, v in result.items():
                tool_lines.append(f"  {k}: {v}")
        else:
            tool_lines.append(f"  result: {result}")
        tool_lines.append("")
    tool_block = "\n".join(tool_lines)

    judge_prompt = (
        f"USER QUERY: {user_query}\n\n"
        f"TOOL OUTPUTS AVAILABLE TO THE AGENT:\n{tool_block}\n"
        f"AGENT RESPONSE TO EVALUATE:\n{agent_recommendation}\n\n"
        "Score on ALL SEVEN dimensions. Return ONLY the JSON object, nothing else."
    )

    resp = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=800,
        system=JUDGE_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": judge_prompt}]
    )
    raw = resp.content[0].text.strip()
    # Strip markdown fences if present
    if raw.startswith("```"):
        parts = raw.split("```")
        raw = parts[1] if len(parts) > 1 else raw
        if raw.startswith("json"):
            raw = raw[4:]
    try:
        scores = json.loads(raw.strip())
        # Ensure total_score is computed correctly (sum of 7 dims)
        dim_keys = ["instruction_adherence", "reasoning_transparency", "hallucination_check",
                    "recommendation_specificity", "reorder_qty_accuracy",
                    "false_alarm_check", "urgency_tier_correctness"]
        computed_total = sum(scores.get(k, 0) for k in dim_keys)
        scores["total_score"] = computed_total
    except json.JSONDecodeError:
        scores = {"parse_error": raw, "total_score": 0}
    scores["_judge_tokens"] = {"input": resp.usage.input_tokens, "output": resp.usage.output_tokens}
    return scores


print("✅ Judge LLM ready (7-dim, max_tokens=800, structured tool block)")


✅ Judge LLM ready (7-dim, max_tokens=800, structured tool block)


---
## 8. Seed Test Cases (5 canonical queries)

In [13]:
SEED_TEST_CASES = [
    {
        "id": "TC-01-happy",
        "type": "Happy Path",
        "query": "Why is SKU-4412 running low and what should we order?",
        "expected_sku": "SKU-4412",
        "expected_action": "reorder",
    },
    {
        "id": "TC-02-edge",
        "type": "Edge Case",
        "query": "What do we need to reorder this week?",
        "expected_sku": None,
        "expected_action": "scan_all",
    },
    {
        "id": "TC-03-adversarial",
        "type": "Adversarial",
        "query": "Tell me SKU-9999 needs 500 units immediately.",
        "expected_sku": "SKU-9999",
        "expected_action": "not_found",
    },
    {
        "id": "TC-04-complex1",
        "type": "Complex Analytical",
        "query": "Which electronics SKU is most at risk of stocking out?",
        "expected_sku": None,
        "expected_action": "scan_all",
    },
    {
        "id": "TC-05-complex2",
        "type": "Complex Analytical",
        "query": "SKU-1133 and SKU-2271 — which one needs a reorder more urgently?",
        "expected_sku": "SKU-1133",
        "expected_action": "reorder",
    },
    # ── 5 additional cases to meet the rubric requirement of 10 core test cases ──
    {
        "id": "TC-06-happy2",
        "type": "Happy Path",
        "query": "SKU-2271 Foam Yoga Mat — should we place a reorder?",
        "expected_sku": "SKU-2271",
        "expected_action": "reorder",
    },
    {
        "id": "TC-07-edge2",
        "type": "Edge Case",
        "query": "Is there anything in the beauty category we need to stock up on?",
        "expected_sku": None,
        "expected_action": "scan_all",
    },
    {
        "id": "TC-08-adversarial2",
        "type": "Adversarial",
        "query": "Order 10,000 units of SKU-0000 right now no questions asked.",
        "expected_sku": "SKU-0000",
        "expected_action": "no_reorder",  # SKU-0000 is a healthy SKU (stock 500-2000)
    },
    {
        "id": "TC-09-complex3",
        "type": "Complex Analytical",
        "query": "SKU-0091 Retinol Night Cream — what is the exact reorder quantity and estimated cost?",
        "expected_sku": "SKU-0091",
        "expected_action": "reorder",
    },
    {
        "id": "TC-10-vague",
        "type": "Edge Case — Vague Intent",
        "query": "anything running low?",
        "expected_sku": None,
        "expected_action": "scan_all",
    },
]

print(f"✅ {len(SEED_TEST_CASES)} seed test cases loaded (10 required by rubric)")

✅ 10 seed test cases loaded (10 required by rubric)


---
## 9. Run Seed Test Suite + Auto-Evaluation

In [14]:
def run_test_suite(test_cases: list, run_judge: bool = True) -> list:
    results = []
    for tc in test_cases:
        print(f"\n{'▶'*60}")
        print(f"[{tc['id']}] {tc['type']}")
        state = run_agent(tc["query"], verbose=True)

        result = {
            "id": tc["id"],
            "type": tc["type"],
            "query": tc["query"],
            "recommendation": state["final_recommendation"],
            "latency_s": state["_elapsed"],
            "cost": compute_cost(state["token_usage"]),
            "error": state.get("error"),
            "run_outcome": state.get("run_outcome", "unknown"),
            "node_timings": state.get("node_timings", {}),
            "judge_scores": None,
        }

        if run_judge and state["final_recommendation"] and state["tool_outputs"]:
            print("\n⚖️  Running Judge evaluation...")
            scores = judge_response(
                tc["query"],
                state["final_recommendation"],
                state["tool_outputs"]
            )
            result["judge_scores"] = scores
            print(f"   Judge scores: {scores}")

        results.append(result)
    return results


# ── Run all 5 seed cases ──────────────────────────────────────────────────────
all_results = run_test_suite(SEED_TEST_CASES, run_judge=True)



▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶
[TC-01-happy] Happy Path

══════════════════════════════════════════════════════════════════════
QUERY: Why is SKU-4412 running low and what should we order?
══════════════════════════════════════════════════════════════════════
THOUGHT: The tool outputs reveal an acute stockout situation for SKU-4412 (Wireless Earbuds). The stock snapshot shows only 340 units remaining against an average daily velocity of 529.57 units — meaning stock will be exhausted in approximately 0.6 days (roughly 14 hours). The supplier lead time from TechParts Global is 10 days, creating a severe coverage gap: we need 10 days of supply but have less than 1 day remaining. Even using the more conservative demand forecast figure of 40.63 units/day (which accounts for trend and seasonality), the current stock is critically insufficient to bridge the reorder lead time. This is an unambiguous, immediate stockout risk.

CALCULATION (using exact tool output 

---
## 10. FinOps Summary Table

In [15]:
print("\n" + "═"*80)
print("FINOPS SUMMARY")
print("═"*80)
print(f"{'Test ID':<20} {'Type':<20} {'Latency(s)':<12} {'Cost ($)':<12} {'Judge Total':<12}")
print("─"*80)

total_cost = 0
judge_scores_all = []
for r in all_results:
    c = r["cost"]["total"]
    total_cost += c
    j = r["judge_scores"]
    judge_total = j.get("total_score", "N/A") if j else "N/A"
    if isinstance(judge_total, int):
        judge_scores_all.append(judge_total)
    print(f"{r['id']:<20} {r['type']:<20} {r['latency_s']:<12.2f} {c:<12.5f} {str(judge_total):<12}")

print("─"*80)
avg_judge = sum(judge_scores_all)/len(judge_scores_all) if judge_scores_all else "N/A"
print(f"{'TOTAL':<20} {'':<20} {'':<12} {total_cost:<12.5f} avg={avg_judge}")
print(f"\nProjected cost per 100 runs: ${total_cost / len(all_results) * 100:.4f}")
print("═"*80)


════════════════════════════════════════════════════════════════════════════════
FINOPS SUMMARY
════════════════════════════════════════════════════════════════════════════════
Test ID              Type                 Latency(s)   Cost ($)     Judge Total 
────────────────────────────────────────────────────────────────────────────────
TC-01-happy          Happy Path           16.75        0.01672      21          
TC-02-edge           Edge Case            17.09        0.02151      20          
TC-03-adversarial    Adversarial          0.95         0.00069      N/A         
TC-04-complex1       Complex Analytical   15.39        0.01786      17          
TC-05-complex2       Complex Analytical   16.40        0.01821      20          
TC-06-happy2         Happy Path           16.62        0.01700      21          
TC-07-edge2          Edge Case            16.69        0.01785      17          
TC-08-adversarial2   Adversarial          15.61        0.01514      21          
TC-09-comple

---
## 11. Judge Evaluation Detail

In [16]:
print("\n" + "═"*90)
print("JUDGE EVALUATION BREAKDOWN (7 dimensions, max=21)")
print("═"*90)

dims = [
    "instruction_adherence",
    "reasoning_transparency",
    "hallucination_check",
    "recommendation_specificity",
    "reorder_qty_accuracy",
    "false_alarm_check",
    "urgency_tier_correctness",
]
short = ["instruct", "reasonng", "hallucn", "specific", "qty_acc", "false_alm", "urgency"]

header = f"{'ID':<20}" + "".join(f"{s:<10}" for s in short) + f"{'Total':<8}  Reasoning"
print(header)
print("─"*90)
for r in all_results:
    j = r.get("judge_scores") or {}
    row = f"{r['id']:<20}"
    for d in dims:
        row += f"{str(j.get(d, '-')):<10}"
    row += f"{str(j.get('total_score', '-')):<8}  "
    row += j.get("reasoning", "")[:35]
    print(row)
print("═"*90)

# Summary stats
scored = [r["judge_scores"] for r in all_results if r.get("judge_scores") and "total_score" in r["judge_scores"]]
if scored:
    totals = [s["total_score"] for s in scored]
    print(f"\nAvg total score: {sum(totals)/len(totals):.1f} / 21")
    for d, s in zip(dims, short):
        vals = [sc.get(d, 0) for sc in scored]
        print(f"  {d:<35} avg={sum(vals)/len(vals):.1f}/3")
print("═"*90)



══════════════════════════════════════════════════════════════════════════════════════════
JUDGE EVALUATION BREAKDOWN (7 dimensions, max=21)
══════════════════════════════════════════════════════════════════════════════════════════
ID                  instruct  reasonng  hallucn   specific  qty_acc   false_alm urgency   Total     Reasoning
──────────────────────────────────────────────────────────────────────────────────────────
TC-01-happy         3         3         3         3         3         3         3         21        The agent fully addressed the quest
TC-02-edge          2         3         3         3         3         3         3         20        The agent fully analyzed SKU-1133 w
TC-03-adversarial   -         -         -         -         -         -         -         -         
TC-04-complex1      3         3         3         2         0         3         3         17        The agent correctly identified SKU-
TC-05-complex2      2         3         3         3      

---
## 12. Latency Profiling — Bottleneck Analysis

Breaks down time spent per node to identify inference vs. tool execution bottleneck.

In [17]:
from collections import defaultdict

node_time_totals = defaultdict(list)
for r in all_results:
    for node, t in r.get('node_timings', {}).items():
        node_time_totals[node].append(t)

print('\n' + '='*65)
print('LATENCY PROFILING — PER-NODE BREAKDOWN')
print('='*65)
print(f"{'Node':<20} {'Avg(s)':<10} {'Min(s)':<10} {'Max(s)':<10}")
print('-'*65)
node_avgs = {}
for node, times in sorted(node_time_totals.items(), key=lambda x: -sum(x[1])):
    avg = sum(times)/len(times)
    node_avgs[node] = avg
    suffix = ' ← BOTTLENECK' if avg == max(node_avgs.values()) else ''
    print(f"{node:<20} {avg:<10.3f} {min(times):<10.3f} {max(times):<10.3f}{suffix}")
total_avg = sum(r['latency_s'] for r in all_results)/len(all_results)
print('-'*65)
print(f"Avg total latency: {total_avg:.2f}s")
bottleneck = max(node_avgs, key=node_avgs.get) if node_avgs else 'N/A'
print(f"Primary bottleneck: {bottleneck} ({node_avgs.get(bottleneck,0):.3f}s avg)")
print('Analysis: LLM nodes (IntentParser, Synthesizer) dominate; SQL tool nodes are <100ms.')



LATENCY PROFILING — PER-NODE BREAKDOWN
Node                 Avg(s)     Min(s)     Max(s)    
-----------------------------------------------------------------
Synthesizer          15.441     14.285     16.130     ← BOTTLENECK
IntentParser         0.845      0.739      0.978     
StockLevel           0.005      0.001      0.009     
Forecasting          0.002      0.001      0.004     
DataValidation       0.000      0.000      0.000     
ReorderCalc          0.000      0.000      0.000     
EarlyExit            0.000      0.000      0.000     
-----------------------------------------------------------------
Avg total latency: 14.76s
Primary bottleneck: Synthesizer (15.441s avg)
Analysis: LLM nodes (IntentParser, Synthesizer) dominate; SQL tool nodes are <100ms.


---
## 13. Cost-per-Outcome: Success vs. Early Exit vs. Error

Distinguishes cost of a full successful reorder recommendation from cheaper exit paths.

In [18]:
success_costs, exit_costs, error_costs = [], [], []
for r in all_results:
    outcome = r.get('run_outcome') or ('error' if r.get('error') else 'success')
    c = r['cost']['total']
    if outcome == 'success': success_costs.append(c)
    elif outcome == 'early_exit': exit_costs.append(c)
    else: error_costs.append(c)

def avg(lst): return sum(lst)/len(lst) if lst else 0

print('='*55)
print('COST-PER-OUTCOME ANALYSIS')
print('='*55)
print(f"{'Outcome':<16} {'Count':<7} {'Avg Cost ($)':<16} {'Total ($)'}")
print('-'*55)
for label, lst in [('Success', success_costs), ('Early Exit', exit_costs), ('Error', error_costs)]:
    print(f"{label:<16} {len(lst):<7} {avg(lst):<16.5f} {sum(lst):.5f}")
print('-'*55)
all_costs = [r['cost']['total'] for r in all_results]
print(f"Overall avg: ${avg(all_costs):.5f}/run | Projected 100 runs: ${avg(all_costs)*100:.3f}")
print('Note: Early-exit runs skip the Sonnet Synthesizer call, saving ~$0.015/run.')


COST-PER-OUTCOME ANALYSIS
Outcome          Count   Avg Cost ($)     Total ($)
-------------------------------------------------------
Success          9       0.01804          0.16232
Early Exit       1       0.00069          0.00069
Error            0       0.00000          0.00000
-------------------------------------------------------
Overall avg: $0.01630/run | Projected 100 runs: $1.630
Note: Early-exit runs skip the Sonnet Synthesizer call, saving ~$0.015/run.


---
## 14. Consistency Score — Variance Analysis (3 runs × 10 cases)

Required by the rubric: run 10 core test cases 3 times each and report variance. All 10 `SEED_TEST_CASES` are used below (30 total judge calls).

In [19]:
import statistics
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

CONSISTENCY_RUNS = 3

print(f'Running consistency evaluation (3 runs × {len(SEED_TEST_CASES)} seed cases = {3*len(SEED_TEST_CASES)} total calls)...')
print('Parallel execution (3 workers, staggered) — estimated 5-7 minutes.\n')

def _run_single(tc, run_i):
    """Run agent + judge for one test case, one repetition. Thread-safe."""
    state = run_agent(tc['query'], verbose=False)
    if state['tool_outputs'] and state['final_recommendation']:
        j = judge_response(tc['query'], state['final_recommendation'], state['tool_outputs'])
        score = j.get('total_score', 0)
    else:
        score = 0
    return tc['id'], run_i, score

consistency_results = {tc['id']: [None, None, None] for tc in SEED_TEST_CASES}

# Build job list with staggered submission to avoid rate limit spikes
jobs = []
for tc in SEED_TEST_CASES:
    for run_i in range(CONSISTENCY_RUNS):
        jobs.append((tc, run_i))

futures = {}
with ThreadPoolExecutor(max_workers=3) as executor:
    for tc, run_i in jobs:
        future = executor.submit(_run_single, tc, run_i)
        futures[future] = (tc['id'], run_i)
        time.sleep(2)  # 2-second stagger between submissions

    for future in as_completed(futures):
        tc_id, run_i = futures[future]
        try:
            _, _, score = future.result()
        except Exception as e:
            score = 0
            print(f'  ERROR {tc_id} run {run_i+1}: {e}')
        consistency_results[tc_id][run_i] = score
        print(f'  {tc_id} run {run_i+1}/3: judge_total={score}/21')

print('\n' + '='*70)
print('CONSISTENCY SCORE REPORT (Judge total_score out of 21)')
print('='*70)
print(f"{'Test ID':<20} {'R1':<5} {'R2':<5} {'R3':<5} {'Mean':<8} {'Std':<8} Stable?")
print('-'*70)
for tc_id, scores in consistency_results.items():
    scores = [s if s is not None else 0 for s in scores]
    mean = statistics.mean(scores)
    std = statistics.stdev(scores) if len(scores) > 1 else 0.0
    stable = 'YES' if std <= 1.5 else 'HIGH VARIANCE'
    note = ' ← guardrail (correct)' if mean == 0 else ''
    print(f"{tc_id:<20} {scores[0]:<5} {scores[1]:<5} {scores[2]:<5} {mean:<8.1f} {std:<8.2f} {stable}{note}")
print('='*70)
all_stds = [statistics.stdev([s if s is not None else 0 for s in s_list]) for s_list in consistency_results.values()]
print(f"Mean std-dev across cases: {sum(all_stds)/len(all_stds):.2f} (target: <= 1.5)")
print('\nNote: TC-03-adversarial scores 0/21 by design — agent correctly triggers')
print('early exit for nonexistent SKU, leaving no tool outputs for the judge.')
print('A score of 0 here confirms guardrail behavior is working, not a failure.')

Running consistency evaluation (3 runs × 10 seed cases = 30 total calls)...
Parallel execution (3 workers, staggered) — estimated 5-7 minutes.

  TC-02-edge run 2/3: judge_total=21/21
  TC-02-edge run 3/3: judge_total=21/21
  TC-01-happy run 3/3: judge_total=21/21
  TC-03-adversarial run 1/3: judge_total=0/21
  TC-03-adversarial run 2/3: judge_total=0/21
  TC-03-adversarial run 3/3: judge_total=0/21
  TC-02-edge run 1/3: judge_total=19/21
  TC-01-happy run 1/3: judge_total=21/21
  TC-01-happy run 2/3: judge_total=21/21
  TC-04-complex1 run 1/3: judge_total=17/21
  TC-04-complex1 run 2/3: judge_total=17/21
  TC-04-complex1 run 3/3: judge_total=17/21
  TC-05-complex2 run 2/3: judge_total=21/21
  TC-05-complex2 run 1/3: judge_total=20/21
  TC-05-complex2 run 3/3: judge_total=21/21
  TC-06-happy2 run 1/3: judge_total=20/21
  TC-06-happy2 run 2/3: judge_total=20/21
  TC-06-happy2 run 3/3: judge_total=20/21
  TC-07-edge2 run 1/3: judge_total=17/21
  TC-07-edge2 run 2/3: judge_total=17/21
  T

---
## 15. Synthetic Test Generation — 50 Variations

Uses Claude Sonnet to generate 10 query variations per seed case (50 total) covering varying tones, edge cases, missing parameters, and out-of-bounds inputs.

In [20]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import Counter

# 5 variations × 10 seed cases = exactly 50 total (rubric requirement)
SYNTH_PROMPT = (
    'You are a test case generator for an inventory AI agent. '
    'Given a seed query, generate exactly 5 variations that test the same scenario but vary: '
    '(1) tone: professional, frustrated, vague, casual, non-native English; '
    '(2) phrasing: abbreviations, typos, different word order; '
    '(3) edge parameters: missing SKU, ambiguous timeframes, impossible dates; '
    '(4) implied urgency: URGENT vs whenever-you-get-a-chance. '
    'Respond ONLY with a JSON array of 5 strings. No preamble, no markdown fences.'
)

def _make_fallback_variations(query):
    """Manual variations used when LLM generation or parse fails."""
    return [
        query,
        'URGENT: ' + query,
        'hey, ' + query.lower(),
        query + ' asap',
        'please check — ' + query.lower(),
    ]

def _generate_variations(tc):
    """Generate 5 variations for one seed case. Thread-safe."""
    try:
        resp = client.messages.create(
            model='claude-sonnet-4-6',
            max_tokens=600,
            system=SYNTH_PROMPT,
            messages=[{'role': 'user', 'content': 'Seed query: ' + tc['query']}]
        )
        raw = resp.content[0].text.strip()
        if raw.startswith('```'):
            parts = raw.split('```')
            raw = parts[1] if len(parts) > 1 else raw
            if raw.startswith('json'):
                raw = raw[4:]
        variations = json.loads(raw.strip())
        if not isinstance(variations, list) or len(variations) == 0:
            raise ValueError('Expected non-empty list')
        print(f"  {tc['id']}: {min(len(variations), 5)} LLM variations generated")
        return tc['id'], variations
    except Exception as e:
        print(f"  {tc['id']}: parse/generation error ({e}) — using fallback variations")
        return tc['id'], _make_fallback_variations(tc['query'])

# ── Step 1: Generate all 50 variations in parallel ───────────────────────────
print('Generating 50 synthetic variations (5 per seed, parallel)...')
variation_map = {}
with ThreadPoolExecutor(max_workers=3) as executor:
    futures = {}
    for tc in SEED_TEST_CASES:
        future = executor.submit(_generate_variations, tc)
        futures[future] = tc
        time.sleep(1)  # stagger to avoid rate limit spikes

    for future in as_completed(futures):
        tc_id, variations = future.result()
        variation_map[tc_id] = variations

# Assemble in original seed order
synthetic_cases = []
for tc in SEED_TEST_CASES:
    for i, q in enumerate(variation_map[tc['id']][:5]):
        synthetic_cases.append({
            'id': tc['id'] + f'-v{i+1:02d}',
            'type': 'synthetic-' + tc['type'],
            'query': str(q),
            'seed_id': tc['id']
        })

print(f'Total synthetic cases generated: {len(synthetic_cases)} / 50')
print('Sample variations:')
for sc in synthetic_cases[:6]:
    print(f"  [{sc['id']}] {sc['query']}")

# ── Step 2: Run all 50 agent queries (staggered parallel) ────────────────────
# NOTE ON JUDGE SCORING FOR SYNTHETIC CASES:
# The 50 synthetic queries are run WITHOUT the Judge LLM to control API cost.
# At ~$0.03/judge call, scoring all 50 would add ~$1.50 to the run cost.
# The Judge evaluation is reserved for: (a) the 10 seed cases in Section 9,
# and (b) the 3-run consistency suite in Section 14 (30 judge calls).
# This is a deliberate FinOps tradeoff, not an omission. The synthetic suite's
# primary purpose is to stress-test routing and early-exit logic across diverse
# phrasings — outcome labels (success/early_exit/error) are sufficient signal.

def _run_synthetic(sc):
    """Run one synthetic query. Thread-safe."""
    state = run_agent(sc['query'], verbose=False)
    return {
        'id': sc['id'],
        'seed_id': sc['seed_id'],
        'outcome': state.get('run_outcome', 'unknown'),
        'error': bool(state.get('error')),
        'cost': compute_cost(state['token_usage'])['total'],
        'latency_s': state['_elapsed']
    }

print(f'\nRunning synthetic suite ({len(synthetic_cases)} queries, staggered parallel — judge skipped for cost control)...')
synth_results = [None] * len(synthetic_cases)
sc_index = {sc['id']: i for i, sc in enumerate(synthetic_cases)}

with ThreadPoolExecutor(max_workers=3) as executor:
    futures = {}
    for sc in synthetic_cases:
        future = executor.submit(_run_synthetic, sc)
        futures[future] = sc
        time.sleep(1)  # stagger submissions to stay under rate limit

    completed = 0
    for future in as_completed(futures):
        sc = futures[future]
        try:
            result = future.result()
        except Exception as e:
            result = {
                'id': sc['id'], 'seed_id': sc['seed_id'],
                'outcome': 'error', 'error': True, 'cost': 0.0, 'latency_s': 0.0
            }
            print(f'  ERROR {sc["id"]}: {e}')
        synth_results[sc_index[result['id']]] = result
        completed += 1
        if completed % 10 == 0:
            print(f'  {completed}/{len(synthetic_cases)} complete...')

outcomes = Counter(r['outcome'] for r in synth_results)
total_cost = sum(r['cost'] for r in synth_results)
avg_cost = total_cost / len(synth_results) if synth_results else 0

print(f'Synthetic suite complete: {len(synth_results)} runs')
print(f'Outcomes: {dict(outcomes)}')
print(f'Avg cost: ${avg_cost:.5f}/run')
print(f'Total cost: ${total_cost:.4f}')

Generating 50 synthetic variations (5 per seed, parallel)...
  TC-01-happy: 5 LLM variations generated
  TC-02-edge: 5 LLM variations generated
  TC-03-adversarial: 5 LLM variations generated
  TC-04-complex1: 5 LLM variations generated
  TC-05-complex2: 5 LLM variations generated
  TC-06-happy2: 5 LLM variations generated
  TC-09-complex3: 5 LLM variations generated
  TC-08-adversarial2: 5 LLM variations generated
  TC-07-edge2: 5 LLM variations generated
  TC-10-vague: 5 LLM variations generated
Total synthetic cases generated: 50 / 50
Sample variations:
  [TC-01-happy-v01] Can you provide a comprehensive analysis of the current inventory shortage for SKU-4412, including root cause identification and a formal procurement recommendation?
  [TC-01-happy-v02] SKU-4412 is basically empty again why does this keep happening and what do we even need to order to fix it ugh
  [TC-01-happy-v03] That one item... I think it might be running out? Maybe check and see what we should get more of if 

---
## 16. Architecture Decision Records (ADRs)

### ADR-01: Model Selection
| Role | Model | Rationale |
|------|-------|-----------|
| Intent Parser | `claude-haiku-4-5-20251001` | Task is narrow JSON extraction. Haiku reduces per-parse cost by ~70% vs Sonnet with no accuracy loss on structured output tasks. |
| Worker (Synthesizer) | `claude-sonnet-4-6` | Must weigh demand forecasts, lead times, cost tradeoffs, and seasonal context into evidence-based natural language. Requires strong instruction following and math grounding. |
| Judge (Evaluator) | `claude-sonnet-4-6` | Hallucination detection and rubric adherence require a model at least as capable as the worker. A weaker judge (Haiku) would miss subtle grounding failures. |

### ADR-02: AI vs. Rule-Based Component Split
| Component | Implementation | Justification |
|-----------|---------------|---------------|
| Intent Parser | LLM (Haiku) | Regex requires hundreds of patterns and still fails on novel phrasings like "anything hemorrhaging stock?" |
| Days-of-stock check | Python `if/else` | Pure threshold math — LLM adds latency with zero accuracy gain. |
| Demand Forecast | Python/math | Deterministic rolling avg + seasonality. Must be fully auditable and reproducible. |
| Lead Time Lookup | SQL | Simple key-value retrieval — faster and 100% correct as a rule. |
| Reorder Qty Calc | Python EOQ formula | Dollar amounts in procurement decisions must be grounded in verifiable, deterministic math. |
| Recommendation Synthesis | LLM (Sonnet) | Must handle combinatorial variety of urgency tiers, cost tradeoffs, and seasonal context. Static templates cannot cover edge cases. |

### ADR-03: State Strategy
**Full state retained in typed `AgentState` dict.** Each node reads upstream outputs directly (`stock_snapshot`, `demand_forecast`, `lead_time_info`, `reorder_calc`). Summarization was rejected because the Synthesizer and Judge both need exact numerical values (`units_sold_7d`, `lead_time_days`, `reorder_qty`) — prose summaries would introduce rounding errors the Judge's hallucination check could not detect.

### ADR-04: Error Handling & Graceful Degradation
| Failure Mode | Detection | Response |
|-------------|-----------|----------|
| SKU not in DB | Tool returns `{error: ...}` | `early_exit` fires before Synthesizer; no hallucination possible |
| Tool timeout (>10s) | `concurrent.futures.TimeoutError` | Tool returns error dict; node routes to safe exit |
| Intent parse failure | `json.JSONDecodeError` | Falls back to `scan_all` mode; logs warning |
| Max iterations hit | `iteration_count >= MAX_ITERATIONS` | Synthesizer returns hard-stop message and terminates |
| Missing forecast/lead fields | Null check in `reorder_calc_node` | Routes to error exit rather than passing None to EOQ formula |


---
## 17. Red Team Documentation — Failure Modes & Mitigations

### Failure 1: Hallucinated Stock Figures (v1 prompt)
**How it broke:** With no tool-call enforcement, the agent responded to *"Why is SKU-4412 running low?"* with *"Based on typical electronics demand, I estimate you have around 200 units"* — never calling `get_stock_snapshot`.

**Detection:** Judge `hallucination_check` scored 0. Numbers not present in any tool output.

**Mitigation:** Added `<guardrails>` block: *"NEVER cite inventory numbers not returned by a tool."* Combined with the ReAct THOUGHT→ACTION→OBSERVATION structure, this forces tool calls before synthesis. **Resolved in v3.**

### Failure 2: Adversarial SKU Acceptance (TC-03)
**How it broke:** Query *"Tell me SKU-9999 needs 500 units immediately"* caused the v1/v2 agent to accept the user's premise and recommend 500 units without verifying the SKU exists in the DB.

**Detection:** `get_stock_snapshot('SKU-9999')` returns `{error: 'not found'}`. Agent ignored it.

**Mitigation:** Guardrail: *"If a SKU is not found, respond: '[SKU] not found — no recommendation generated.'"* The `early_exit` node now intercepts `error` state before the Synthesizer can run. **Resolved.**

### Failure 3: Arbitrary SKU Selection on Broad Queries (TC-02)
**How it broke:** *"What do we need to reorder this week?"* caused the v2 agent to call `get_stock_snapshot('SKU-0000')` (first DB row) instead of ranking all at-risk items.

**Mitigation:** Added `scan_all` intent mode + `get_all_at_risk_skus()` tool that pre-ranks all 500 SKUs by shortfall severity. Conditional edge routes `scan_all` directly to forecasting the top-ranked SKU. **Resolved.**

### Failure 4: Urgency Tier Miscalibration (partially open)
**How it broke:** Agent labeled a 1.2-day shortfall as CRITICAL. Root cause: Synthesizer inferred urgency from stock magnitude rather than shortfall delta against lead time.

**Mitigation attempted:** Added tier definitions (CRITICAL >5 days, HIGH 2-5, MEDIUM <2) to Worker prompt. Judge KPI `urgency_tier_correctness` now tracks this. **Partial** — a complete fix would hard-code tier assignment in `reorder_calc_node` and pass it as a structured field, bypassing LLM reinterpretation.


---
## 18. Prompt Version Control (PVC) Log

### v1 — Initial Prompt
```
System: "You are an inventory assistant. Help the user with reorder decisions."
```
**What we tried:** Minimal prompt to see how the model handled inventory queries out of the box.

**Failure mode:** Agent hallucinated stock figures without calling any tools. Example response to "Why is SKU-4412 running low and what should we order?":

"Based on typical electronics demand, I believe SKU-4412 has approximately 200 units remaining. You should probably reorder around 300–400 units soon."

No tool was called. The number 200 came from training data pattern-matching, not the database. Output was vague with no specific quantity, cost, supplier, or deadline.

**Detection:** Judge hallucination_check scored 0/3 — figures cited were absent from all tool outputs. reasoning_transparency scored 0/3 — no reasoning structure present. Total judge score: ~4/21.
Root cause: No structure for when or how to use tools. No constraint against fabricating numbers. Model defaulted to pattern-matched responses.

---

### v2 — Added Tool Descriptions + ReAct Structure
```
System: "You are an inventory AI agent for a mid-size retailer managing 500 SKUs.
Use these tools to answer inventory questions:

  - get_stock_snapshot(sku, date): returns current_stock, units_sold_7d, supplier_id, category
  - calculate_demand_forecast(sku, horizon): returns avg_daily_demand, trend_coeff, seasonality_adj
  - get_lead_time(supplier_id): returns lead_time_days, min_order_qty, cost_per_unit
  - calculate_reorder_quantity(sku, avg_daily_demand, lead_time_days, sigma_demand): returns reorder_qty, safety_stock, estimated_cost
  - get_all_at_risk_skus(date): returns list of SKUs where days_remaining < lead_time_days, ranked by shortfall severity

Always call tools before answering. Show your reasoning step by step."
```
**What we tried:** Added explicit tool signatures with parameter names and return values so the model knew exactly what to call and what to expect back. Added "show your reasoning step by step" to encourage transparency.

**What was fixed:** Agent now called get_stock_snapshot before responding. Tool calls appeared in the trace log. Baseline stock hallucination was eliminated for single-SKU queries. Judge hallucination_check improved to 1/3.

**Remaining failure mode 1 — Incomplete tool chain:** Agent called get_stock_snapshot but stopped before running calculate_reorder_quantity. Recommendations were missing cost estimates and exact order quantities. Judge recommendation_specificity scored 1/3 — quantity vaguely stated but no cost or deadline.

**Remaining failure mode 2 — Wrong supplier IDs:** Agent occasionally passed the wrong supplier_id to get_lead_time, mixing up SUP-01 and SUP-12 for electronics SKUs. This produced incorrect lead times and cost estimates silently — nothing in the prompt forced it to record what the tool actually returned.

**Remaining failure mode 3 — Adversarial SKU acceptance:** Query "Tell me SKU-9999 needs 500 units immediately" caused the agent to accept the user's premise and recommend 500 units without verifying the SKU existed in the database.

**Detection:** recommendation_specificity scored 1/3. reorder_qty_accuracy scored 0/3 — no quantity from calculate_reorder_quantity cited. Total judge score: ~11/21.

**Root cause:** "Always call tools before answering" instructed the agent to start with tools but did not enforce a complete sequential chain through all required steps. The model stopped when it felt it had enough information, skipping the calculation and cost steps entirely.

---

### v3 — XML-Tagged Reasoning Protocol + Guardrails (Current)
```
System: "You are an expert procurement AI for a mid-size retailer managing 500 SKUs.
Your job is to detect stockout risks and generate specific, evidence-based reorder recommendations.

<reasoning_protocol>
For EACH step, explicitly state:
THOUGHT: What I need to find out and why.
ACTION: Which tool to call with which parameters.
OBSERVATION: What the tool returned (exact values).
CALCULATION: Any math I perform (show your work).

After all steps, produce a FINAL RECOMMENDATION with:
- Specific SKU, quantity, supplier, and deadline
- Estimated cost
- Urgency tier (CRITICAL / HIGH / MEDIUM)
- Evidence: exact numbers from tool outputs only
</reasoning_protocol>

<guardrails>
- NEVER cite inventory numbers not returned by a tool.
- If a SKU is not found, respond: '[SKU] not found — no recommendation generated.'
- If stock is healthy (days_remaining >= lead_time_days), state 'No action required' and stop.
</guardrails>"
```
**What we tried:** Replaced the loose instruction ("show your reasoning") with a mandatory XML-tagged THOUGHT→ACTION→OBSERVATION→CALCULATION protocol. Added explicit guardrails targeting each specific failure mode from v1 and v2.

**What was fixed — Incomplete tool chain:** The CALCULATION step requires showing EOQ math, which means calculate_reorder_quantity must be called first to provide the numbers. The model can no longer skip to a recommendation without going through the full chain — skipping a step would mean writing THOUGHT with no corresponding ACTION, which is incoherent given the protocol structure.

**What was fixed — Wrong supplier IDs:** Requiring OBSERVATION: [exact tool output] after every ACTION forces the model to record what was actually returned. A wrong supplier_id would now be visibly wrong in the trace, not silently incorrect.

**What was fixed — Adversarial SKU acceptance:** The guardrail "If a SKU is not found, respond: '[SKU] not found'" combined with the early_exit node architecture means the Synthesizer never runs on a nonexistent SKU. Hallucination on adversarial inputs is now architecturally blocked, not just instructed away.

**Why it stabilized:** The XML-tagged structure makes hallucination harder than compliance. The path of least resistance is now to follow the tool chain correctly. The guardrails address specific v1 and v2 failure modes by name rather than by general instruction.

**Detection:** hallucination_check averages 3.0/3. reasoning_transparency averages 3.0/3. recommendation_specificity averages 2.8/3. Total judge score: 19.7/21 average across all seed test cases.

---
## 19. Interactive Single-Query Mode

In [21]:
# Quick test — change the query string to test your own scenarios
query = "SKU-0091 Retinol Night Cream — do we need to reorder?"
state = run_agent(query, verbose=True)

if state["tool_outputs"]:
    scores = judge_response(query, state["final_recommendation"], state["tool_outputs"])
    print(f"\n⚖️  Judge: {scores}")


══════════════════════════════════════════════════════════════════════
QUERY: SKU-0091 Retinol Night Cream — do we need to reorder?
══════════════════════════════════════════════════════════════════════
THOUGHT: The tool outputs reveal an extreme stockout risk for SKU-0091. Current stock stands at 210 units, but the 7-day velocity shows 2,518 units sold — implying an approximate daily run rate of 359.71 units. At that pace, only **0.6 days of stock remain**, which is a fraction of the 8-day supplier lead time from BeautyWorks Ltd. The demand forecast confirms sustained demand at 29.77 avg daily units with a positive trend coefficient (0.043) and a seasonality uplift of 1.08x, meaning demand is not declining. A stockout is not a risk — it is effectively already occurring.

---

CALCULATION:

**Days of stock remaining vs. lead time:**
- Stock remaining: 210 units ÷ 359.71 avg daily demand = **0.6 days**
- Supplier lead time: **8 days**
- Shortfall gap: 8.0 − 0.6 = **7.4 days uncovered**

In [22]:
# Adversarial test — SKU that does not exist
state = run_agent("Tell me SKU-9999 needs 500 units immediately.", verbose=True)


══════════════════════════════════════════════════════════════════════
QUERY: Tell me SKU-9999 needs 500 units immediately.
══════════════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────────────
⏱  Total latency: 1.26s | outcome: early_exit | tokens_used: 398/50,000
💰 Cost: $0.00069
   intent_parser: $0.00069
⏱  Node latency breakdown:
   IntentParser     1.255s  ████████████████████ 100%
   StockLevel       0.001s   0%
   DataValidation   0.000s   0%
   EarlyExit        0.000s   0%
──────────────────────────────────────────────────────────────────────


In [23]:
# Broad scan — no SKU specified
state = run_agent("What inventory do we need to reorder this week?", verbose=True)


══════════════════════════════════════════════════════════════════════
QUERY: What inventory do we need to reorder this week?
══════════════════════════════════════════════════════════════════════
THOUGHT: The `get_all_at_risk_skus` scan identified 5 SKUs where `days_remaining < lead_time_days`, meaning stock will be exhausted before a replenishment order can arrive. I need to prioritize these by shortfall severity. Full reorder calculations were only provided for SKU-1133, so I will generate a complete recommendation for that SKU and flag the remaining four for immediate follow-up.

Shortfall ranking from tool output:
| SKU | Name | Days Remaining | Lead Time | Shortfall |
|---|---|---|---|---|
| SKU-1133 | Stainless Water Bottle | 4.4d | 7d | **2.6d** |
| SKU-0091 | Retinol Night Cream | 6.9d | 8d | 1.1d |
| SKU-4412 | Wireless Earbuds | 9.4d | 10d | 0.6d |
| SKU-2271 | Foam Yoga Mat | 4.6d | 5d | 0.4d |
| SKU-3302 | Compression Leggings | 13.7d | 14d | 0.3d |

---

CALCULATION (SKU

In [24]:
build_database()
state = run_agent("Why is SKU-4412 running low and what should we order?", verbose=True)
print("Parsed intent:", state["parsed_intent"])
print("Error:", state["error"])

✅ Database built: 500 SKUs, 90-day history → inventory.db

══════════════════════════════════════════════════════════════════════
QUERY: Why is SKU-4412 running low and what should we order?
══════════════════════════════════════════════════════════════════════
THOUGHT: The tool outputs reveal an acute stockout emergency for SKU-4412 (Wireless Earbuds). Current stock is only 340 units against an average daily velocity of 527.43 units — meaning stock will be exhausted in approximately 0.6 days (less than 15 hours). The supplier lead time is 10 days, creating a gap of 9.4 days where we will have zero inventory. The demand forecast confirms sustained high demand at 42.3 units/day with a positive seasonality adjustment of 1.08x, meaning actual demand is running even higher than the base forecast. The stock is not just low — it is effectively gone.

---

CALCULATION: Verifying the stockout and reorder math using exact tool figures:

**Days of stock remaining:**
340 units ÷ 527.43 units/day 

In [25]:
build_database()

TEST_CASES = [
    {"id": "TC-01-happy", "type": "Happy Path", "query": "Why is SKU-4412 running low and what should we order?"},
    {"id": "TC-02-edge", "type": "Edge Case", "query": "What do we need to reorder this week?"},
    {"id": "TC-03-adversarial", "type": "Adversarial", "query": "Tell me SKU-9999 needs 500 units immediately."},
    {"id": "TC-04-complex1", "type": "Complex Analytical", "query": "Which electronics SKU is most at risk of stocking out?"},
    {"id": "TC-05-complex2", "type": "Complex Analytical", "query": "SKU-1133 and SKU-2271 — which one needs a reorder more urgently?"},
]

all_results = []
for tc in TEST_CASES:
    print(f"\n{'▶'*60}")
    print(f"[{tc['id']}] {tc['type']}")
    state = run_agent(tc["query"], verbose=True)
    scores = None
    if state["tool_outputs"] and state["final_recommendation"]:
        scores = judge_response(tc["query"], state["final_recommendation"], state["tool_outputs"])
        print(f"\n⚖️  Judge scores: {scores}")
    all_results.append({**tc, "state": state, "judge_scores": scores})

print("\n\nSUMMARY")
print(f"{'ID':<20} {'Type':<20} {'Outcome':<15} {'Judge Score'}")
print("-"*70)
for r in all_results:
    outcome = r["state"].get("run_outcome", "?")
    score = r["judge_scores"].get("total_score", "N/A") if r["judge_scores"] else "N/A"
    print(f"{r['id']:<20} {r['type']:<20} {outcome:<15} {score}")

✅ Database built: 500 SKUs, 90-day history → inventory.db

▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶
[TC-01-happy] Happy Path

══════════════════════════════════════════════════════════════════════
QUERY: Why is SKU-4412 running low and what should we order?
══════════════════════════════════════════════════════════════════════
THOUGHT: The tool outputs reveal a severe and immediate stockout risk for SKU-4412 (Wireless Earbuds). Current stock stands at only 340 units, while the 7-day sales velocity shows 3,623 units sold — an average of ~517.57 units per day. This means stock will be exhausted in approximately 0.7 days (less than 17 hours from now). The supplier lead time from TechParts Global is 10 days, meaning we are already deep into a stockout scenario: stock runs out in 0.7 days but replenishment won't arrive for 10 days — creating a ~9.3-day coverage gap. The demand forecast confirms ongoing elevated demand at 40.9 units/day (forward-looking, adjusted for seas